# Baseline notebook (v1)
Original exploration (best: 86.2% with Logistic Regression + TF-IDF).

One bug fixed: the TF-IDF Naive Bayes cell evaluated a model trained on Bag-of-Words, so its old score (72.75%) was invalid; the correct score is about 66.1%.

The improved, deployable pipeline is in `data_cleaning.py`, `train_model.py` and `app.py` (88.97% on the same test split).

In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("train.txt" , sep=";" , header=None , names=['text','emotion'])
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [4]:
unique_emotion=df["emotion"].unique()
emotion_numbers={}
i=0
for emo in unique_emotion:
    emotion_numbers[emo]=i
    i+=1
df['emotion']=df['emotion'].map(emotion_numbers)    


In [5]:
df.head()


,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [6]:
df['text']=df['text'].apply(lambda x:x.lower())
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [7]:
import string
def remove_punc(txt):
     return txt.translate(str.maketrans(' ',' ',string.punctuation))

In [8]:
df['text']=df['text'].apply(remove_punc)

In [9]:
def remove_numbers(txt):
    new=''
    for i in txt:
        if not i.isdigit():
            new=new+''+i
    return new        
            
    

In [10]:
df['text']=df['text'].apply(remove_numbers)

In [11]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [12]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import nltk

In [14]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [15]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pal16\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pal16\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\pal16\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
stop_words=set(stopwords.words('english'))


In [17]:
def remove(txt):
    words=word_tokenize(txt)
    cleaned=[]
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return '    '.join(cleaned)        

In [18]:
df.columns

Index(['text', 'emotion'], dtype='str')

In [19]:
df['text']=df['text'].apply(remove)

In [20]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful...,0
2,im grabbing minute post feel gr...,1
3,ever feeling nostalgic fireplace k...,2
4,feeling grouchy,1


In [21]:
df.loc[1]['text']

'go    feeling    hopeless    damned    hopeful    around    someone    cares    awake'

In [22]:
#%pip uninstall -y scikit-learn
#%pip install --no-cache-dirscikit-learn    
%pip install scikit-learn
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'],df['emotion'],test_size=0.20, random_state=42)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from sklearn.feature_extraction.text import CountVectorizer ,TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
bow_vectorizer=CountVectorizer()
X_train_bow=bow_vectorizer.fit_transform(X_train)
X_test_bow=bow_vectorizer.transform(X_test)

Tfidf_vectorizer=TfidfVectorizer()
X_train_Tfidf=Tfidf_vectorizer.fit_transform(X_train)
X_test_Tfidf=Tfidf_vectorizer.transform(X_test)


#NB_MODEL

In [27]:
nb_model=MultinomialNB()
nb_model.fit(X_train_bow,y_train)
pred_bow=nb_model.predict(X_test_bow)
print(accuracy_score(y_test,pred_bow))

0.7678125


In [ ]:
nb2_model=MultinomialNB()
nb2_model.fit(X_train_Tfidf,y_train)
pred_Tfidf=nb2_model.predict(X_test_Tfidf)  # FIX: was nb_model (trained on BoW, not TF-IDF)
print(accuracy_score(y_test,pred_Tfidf))

#LOGISTICREGRESSION_MODEL

In [31]:
from sklearn.linear_model import LogisticRegression
log_model= LogisticRegression(max_iter=1000)
log_model.fit(X_train_Tfidf,y_train)
pred_LR=log_model.predict(X_test_Tfidf)
print(accuracy_score(y_test,pred_LR))

0.8615625
